<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/Estadistica/z342_RegLinealNorm_Magicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightGBM con productos mágicos + normalización + imputación

Basado en z403 pero con:
- **LightGBM optimizado con Optuna** en vez de OLS
- **Imputación**: `0` si existía sin venta, `-1` si el producto no existía aún
- **3 normalizaciones** (max, L2, index) antes de armar los lags
- **Fix**: separación correcta entre features de entrenamiento (necesita clase t+2) y de predicción (solo lags)

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle lightgbm optuna

In [ ]:
import os, shutil
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'competencia':     'labo-iii-2026-rosario',
    'periodo_train':   201812,
    'periodo_predict': 201912,
    'lags':            list(range(0, 12)),
    'horizonte':       2,
    'optuna_trials':   50,
    'drive_path':      '/content/buckets/b1/exp/LGBMNormMagicos',
}
os.makedirs(PARAM['drive_path'], exist_ok=True)

PRODUCTOS_MAGICOS = [
    20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
    20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
    20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
    20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
    20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
    20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
    20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
    20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
    20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
    20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
    20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
    20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
    20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
    20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
    20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
    21080, 21088, 21118, 21170, 21200
]
print(f'{len(PRODUCTOS_MAGICOS)} productos mágicos')

# Datos + grilla + imputación

In [ ]:
dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')

tb_ventas = (
    dataset.group_by('product_id','periodo').agg(pl.col('tn').sum())
    .join(tb_apredecir, on='product_id', how='inner')
    .sort(['product_id','periodo'])
)
productos = tb_apredecir['product_id'].to_list()
periodos  = sorted(tb_ventas['periodo'].unique().to_list())

# grilla completa
grilla = pl.DataFrame({
    'product_id': [p for p in productos for _ in periodos],
    'periodo':    periodos * len(productos),
})

primer_periodo = (
    tb_ventas.filter(pl.col('tn') > 0)
    .group_by('product_id').agg(pl.col('periodo').min().alias('primer_periodo'))
)

tb_full = (
    grilla
    .join(tb_ventas.select(['product_id','periodo','tn']), on=['product_id','periodo'], how='left')
    .join(primer_periodo, on='product_id', how='left')
    .with_columns(
        pl.when(pl.col('tn').is_not_null()).then(pl.col('tn'))
          .when(pl.col('periodo') < pl.col('primer_periodo')).then(pl.lit(-1.0))
          .otherwise(pl.lit(0.0)).alias('tn')
    ).drop('primer_periodo')
)

print(f'Grilla: {tb_full.height:,} filas')
print(f'  reales (>0):  {(tb_full["tn"] > 0).sum():,}')
print(f'  ceros:        {(tb_full["tn"] == 0).sum():,}')
print(f'  -1 (no exist):{(tb_full["tn"] == -1).sum():,}')

# Normalizaciones

In [ ]:
def norm_max(serie):
    reales = serie[serie > 0]
    m = float(reales.max()) if len(reales) > 0 else 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / m
    return n, m

def norm_l2(serie):
    reales = serie[serie > 0]
    norma  = float(np.sqrt((reales**2).sum())) if len(reales) > 0 else 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / norma
    return n, norma

def norm_index(serie, n_base=3):
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) > 0 else 1.0
    n = serie.copy()
    n[serie >= 0] = serie[serie >= 0] / base
    return n, base

NORMALIZACIONES = {'sin_norm': None, 'max': norm_max, 'l2': norm_l2, 'index': norm_index}
print('OK')

# Construcción de features

Dos funciones separadas:
- `build_train`: para entrenar — necesita `clase` (t+horizonte), usa solo `periodo_train`
- `build_predict`: para predecir — solo necesita los lags, usa `periodo_predict`

In [ ]:
def get_serie_norm(tb_full, pid, norm_fn):
    df    = tb_full.filter(pl.col('product_id') == pid).sort('periodo')
    serie = df['tn'].to_numpy().astype(float)
    pds   = df['periodo'].to_list()
    if norm_fn is not None:
        serie_norm, escala = norm_fn(serie)
    else:
        serie_norm, escala = serie.copy(), 1.0
    return serie_norm, pds, escala


def build_train(tb_full, productos_train, norm_fn, lags, periodo_train, horizonte):
    """Features de entrenamiento: periodo_train para productos_train.
    Necesita que existan horizonte períodos DESPUÉS del periodo_train."""
    rows = []
    feat_cols = [f'tn_{l}' for l in lags]
    for pid in productos_train:
        serie_norm, pds, _ = get_serie_norm(tb_full, pid, norm_fn)
        if periodo_train not in pds:
            continue
        i = pds.index(periodo_train)
        if i < max(lags) or i + horizonte >= len(serie_norm):
            continue
        row = {f'tn_{l}': float(serie_norm[i - l]) for l in lags}
        row['clase'] = float(serie_norm[i + horizonte])
        rows.append(row)
    tb = pl.DataFrame(rows)
    X  = tb.select(feat_cols).to_numpy()
    y  = tb['clase'].to_numpy()
    return X, y


def build_predict(tb_full, productos, norm_fn, lags, periodo_predict):
    """Features de predicción: periodo_predict para todos los productos.
    NO necesita el período futuro — solo los lags hacia atrás."""
    rows   = []
    escalas = {}
    feat_cols = [f'tn_{l}' for l in lags]
    for pid in productos:
        serie_norm, pds, escala = get_serie_norm(tb_full, pid, norm_fn)
        escalas[pid] = escala
        if periodo_predict not in pds:
            continue
        i = pds.index(periodo_predict)
        if i < max(lags):
            continue
        row = {'product_id': pid}
        row.update({f'tn_{l}': float(serie_norm[i - l]) for l in lags})
        rows.append(row)
    tb = pl.DataFrame(rows)
    return tb, escalas

print('OK')

# LightGBM optimizado con Optuna

In [ ]:
from sklearn.model_selection import cross_val_score

def optimizar_lgbm(X, y, n_trials=50):
    """
    Busca los mejores hiperparámetros de LightGBM con Optuna.
    Usa cross-validation 5-fold sobre los datos de entrenamiento.
    """
    def objective(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
            'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 10, 100),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            'verbose':           -1,
        }
        modelo = lgb.LGBMRegressor(**params)
        scores = cross_val_score(modelo, X, y, cv=5, scoring='neg_root_mean_squared_error')
        return -scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    mejor_params = study.best_params
    mejor_params['verbose'] = -1
    print(f'  Mejor RMSE CV: {study.best_value:.4f}')
    print(f'  Params: {mejor_params}')

    modelo_final = lgb.LGBMRegressor(**mejor_params)
    modelo_final.fit(X, y)
    return modelo_final

print('OK')

# Loop principal — entrenar + predecir + submit

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

feat_cols = [f'tn_{l}' for l in PARAM['lags']]
resultados = {}

for nombre_norm, fn in NORMALIZACIONES.items():
    print(f'\n══ norm={nombre_norm} ══')

    # ── entrenamiento ──
    X_train, y_train = build_train(
        tb_full, PRODUCTOS_MAGICOS, fn,
        PARAM['lags'], PARAM['periodo_train'], PARAM['horizonte']
    )
    print(f'  filas entrenamiento: {len(X_train)}')

    modelo = optimizar_lgbm(X_train, y_train, n_trials=PARAM['optuna_trials'])

    # ── predicción ──
    tb_pred_feat, escalas = build_predict(
        tb_full, productos, fn,
        PARAM['lags'], PARAM['periodo_predict']
    )
    print(f'  productos con lags completos: {tb_pred_feat.height}')

    X_pred   = tb_pred_feat.select(feat_cols).to_numpy()
    pred_norm = modelo.predict(X_pred)

    # desnormalizar
    pids_pred = tb_pred_feat['product_id'].to_list()
    preds_real = [
        {'product_id': pid, 'tn': max(float(pn) * escalas.get(pid, 1.0), 0.0)}
        for pid, pn in zip(pids_pred, pred_norm)
    ]
    tb_pred = pl.DataFrame(preds_real)

    # fallback: mediana 12m para los que no tienen lags completos
    pids_con_pred = set(pids_pred)
    fallback = []
    for pid in productos:
        if pid not in pids_con_pred:
            serie = tb_full.filter(
                (pl.col('product_id') == pid) & (pl.col('tn') >= 0)
            ).sort('periodo').tail(12)['tn'].to_numpy().astype(float)
            fallback.append({'product_id': pid, 'tn': max(float(np.median(serie)), 0.0)})

    tb_final = pl.concat([tb_pred] + ([pl.DataFrame(fallback)] if fallback else [])).sort('product_id')
    resultados[nombre_norm] = tb_final

    print(f'  fallback: {len(fallback)} productos')
    print(f'  total predicciones: {tb_final.height}')

    archivo = f'lgbm_norm_{nombre_norm}.csv'
    mensaje = f'LGBM Optuna norm={nombre_norm} magicos'
    tb_final.write_csv(archivo)
    shutil.copy(archivo, f"{PARAM['drive_path']}/{archivo}")
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'  submitted + guardado: {archivo}')

# Comparación de predicciones entre normalizaciones

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(resultados), figsize=(14, 4))
colores   = ['gray', 'tomato', 'green', 'purple']

for i, (nombre, tb) in enumerate(resultados.items()):
    vals = tb['tn'].to_numpy()
    cap  = np.percentile(vals, 95)
    axes[i].hist(vals[vals <= cap], bins=40, color=colores[i], edgecolor='white', alpha=0.8)
    axes[i].set_title(f'norm={nombre}\nmedia={vals.mean():.2f}  mediana={np.median(vals):.2f}', fontsize=8)
    axes[i].set_xlabel('tn predicho')

plt.suptitle('Distribución predicciones 202002', fontsize=10)
plt.tight_layout()
plt.show()